# 使用scGPT进行多组学数据分析

本教程演示如何使用scGPT模型进行多组学数据分析。多组学数据整合可以同时分析基因表达、染色质可及性、蛋白质丰度等多种组学数据，提供更全面的细胞状态视图。


In [1]:
# 导入必要的库
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import torch
import matplotlib.pyplot as plt

# 添加scGPT路径
sys.path.insert(0, "../")
import scgpt as scg
from scgpt.model import MultiomicModel

print("✅ 所有库加载完成！")

In [2]:
# 设置参数
model_dir = Path("../save/scGPT_multiomic")
data_dir = Path("../data/multiomics")
save_dir = Path("../results/multiomics")

# 创建保存目录
save_dir.mkdir(parents=True, exist_ok=True)

print("✅ 参数设置完成！")

In [3]:
# 加载多组学数据
# 基因表达数据
rna_adata = sc.read_h5ad(data_dir / "rna_data.h5ad")
# 染色质可及性数据
atac_adata = sc.read_h5ad(data_dir / "atac_data.h5ad")
# 蛋白质数据（可选）
protein_adata = sc.read_h5ad(data_dir / "protein_data.h5ad")

print(f"RNA数据: {rna_adata.shape}")
print(f"ATAC数据: {atac_adata.shape}")
print(f"蛋白质数据: {protein_adata.shape}")

print("✅ 多组学数据加载完成！")

In [4]:
# 创建多组学模型
model = MultiomicModel(
    rna_vocab_size=len(rna_adata.var),
    atac_vocab_size=len(atac_adata.var),
    protein_vocab_size=len(protein_adata.var),
    d_model=512,
    nhead=8,
    nlayers=6,
)

# 加载预训练权重
model_file = model_dir / "best_model.pt"
model.load_state_dict(torch.load(model_file, map_location="cpu"))

# 将模型移动到设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print("✅ 多组学模型加载完成！")

In [5]:
# 获取多组学嵌入
multiomic_embeddings = scg.tasks.get_multiomic_embeddings(
    rna_adata=rna_adata,
    atac_adata=atac_adata,
    protein_adata=protein_adata,
    model=model,
    batch_size=32,
    device=device,
)

# 将嵌入添加到RNA数据中
rna_adata.obsm["X_multiomic"] = multiomic_embeddings

print(f"✅ 多组学嵌入获取完成！形状: {multiomic_embeddings.shape}")

In [6]:
# 可视化多组学嵌入
sc.pp.neighbors(rna_adata, use_rep="X_multiomic")
sc.tl.umap(rna_adata)

# 绘制UMAP图
plt.figure(figsize=(10, 8))
sc.pl.umap(rna_adata, color=["cell_type", "batch"], ncols=2, frameon=False)
plt.suptitle("多组学嵌入可视化")
plt.tight_layout()
plt.show()

print("✅ 可视化完成！")

In [7]:
# 保存结果
rna_adata.write_h5ad(save_dir / "multiomic_embeddings.h5ad")

print("✅ 多组学分析完成！")